<a href="https://colab.research.google.com/github/gitmystuff/INFO5810/blob/main/Module_08-Graph_RAG/Grounded_AI_Graph_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grounded AI: Graph RAG

Your Name

## Getting Started

* Colab - get notebook from gitmystuff INFO5810 repository
* Save a Copy in Drive
* Remove Copy of
* Edit your name in the cell above and the filename
* Clean up Colab Notebooks folder
* Submit shared link

**This week needs one extra thing: a free Groq API key.** Groq runs the model we'll use, `openai/gpt-oss-120b`, and gives every student free access. Get your key from console.groq.com, then in Colab click the key icon on the left sidebar, add a secret named `GROQ_API_KEY`, and paste your key in.

## Getting a Groq API Key and Saving It in Google Colab

You will need a **Groq API key** for our upcoming activities. We will store the key securely in **Google Colab Secrets** rather than putting it directly in our code.

Step 1: Create a Groq Account

1. Go to the **Groq Console**:
   `https://console.groq.com`

2. Sign up or log in. You can use Google, GitHub, or your email account.

Step 2: Create an API Key

1. Open **API Keys** in the Groq Console.

2. Click **Create API Key**.

3. Give your key a name. For example:

   `INFO5810`

4. Create the key.

5. **Copy the API key immediately.**

   Keep this key private. Do not paste it into your notebook, GitHub repository, Canvas submission, or anywhere others can see it.

Step 3: Open Your Google Colab Notebook

Open the Google Colab notebook you will be using for class.

On the **left side of Colab**, find the **key icon** for **Secrets** and click it.

Step 4: Add the Groq Key to Colab Secrets

Click **Add new secret** and enter:

**Name**
```text
GROQ_API_KEY
```

**Value**

Paste the Groq API key you copied earlier.

Make sure **Notebook access** is turned **ON** for `GROQ_API_KEY`.

Step 5: That's It

Your Groq API key is now securely stored in Google Colab.

When our notebooks need the key, we can access it with:

```python
from google.colab import userdata

groq_key = userdata.get("GROQ_API_KEY")
```

You do **not** need to paste your actual API key into the Python code.

Important

**Never share or submit your API key.**

If you accidentally expose your key, delete/revoke it in the Groq Console and create a new one.

## Where we are, and where we're going

Last week we trained a network to predict hidden friendships from a graph's structure. This week we connect that same idea — facts stored in a graph — to something you've likely already built: a question-answering system using embeddings and vector search.

**The objective for this module:** connect AI models to structured knowledge graphs so they answer questions using verified facts instead of guessing.

**The plan, in nine parts:**

0. A cold open — ask a language model something it cannot possibly know, and watch what happens.
1. A quick recap of traditional RAG — embeddings, vector search.
2. The gap — why "sounds similar" isn't the same as "is the right fact."
3. Build a small knowledge graph (NetworkX) — reusing the Bob/Alice/Mona Lisa world from earlier.
4. Retrieval — given a question, search the graph for relevant facts.
5. Grounding — build a prompt that answers only from retrieved facts.
6. Call a real model (Groq, `openai/gpt-oss-120b`) and watch grounding work.
7. The payoff — same question, grounded vs. not, side by side.
8. A quiet bug — retrieval that's too shallow to find the whole answer.
9. Where this leads — the same system, with Neo4j as the backend, later this week.

Same rule as always: **overview, not mastery.**

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

%pip install groq -q
import groq
print("groq SDK ready")

groq SDK ready


---
## Part 0 — A cold open: ask it something it cannot know

Here's a question for a language model: **"What company does Bob work at?"**

There's no way for any model to know this — Bob is a made-up person from our own course examples, and "Meridian Analytics" is a company we're about to invent. Let's ask anyway, with no help at all, and see what happens.

In [ ]:
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")   # fallback if not running in Colab

MODEL = "openai/gpt-oss-120b"

def ask_model(messages, model=MODEL):
    # Calls Groq if a key is available; otherwise returns a clearly-labeled stand-in
    # so the rest of the notebook still runs. Replace nothing here -- just add your key.
    if GROQ_API_KEY:
        client = groq.Groq(api_key=GROQ_API_KEY)
        try:
            resp = client.chat.completions.create(model=model, messages=messages, temperature=0.2)
            return resp.choices[0].message.content
        except groq.GroqError as e:
            return f"[Groq API error: {e}]"
    else:
        return "[OFFLINE FALLBACK -- no GROQ_API_KEY found. Add your key to see the real model's answer.]"

cold_open_answer = ask_model([
    {"role": "user", "content": "What company does Bob work at?"}
])
print(cold_open_answer)

I’m sorry, but I don’t have that information.


**Reading the output.** If your key is set up, you should see the model either admit it doesn't know, or — more interesting, and worth watching for — confidently invent a plausible-sounding answer anyway. Neither is correct, because there's no "Bob" or "Meridian Analytics" to know about. This is the whole problem Graph RAG exists to solve: **a model that can't retrieve real facts has only one option when asked something specific — guess.**

By the end of this notebook, this exact question gets a correct, verified answer, because we'll have given the model somewhere real to look.

---
## Part 1 — A quick recap: traditional RAG

You've already built this with ChromaDB: **R**etrieval-**A**ugmented **G**eneration. The idea in three steps:

1. Break your documents into small chunks of text.
2. Turn each chunk into a vector (an embedding), the same way we turned words into vectors.
3. When a question comes in, turn the question into a vector too, find the chunks whose vectors are *closest* to it, and hand those chunks to the model as context.

We'll do a quick version of this with five short sentences about Bob, Alice, and the Mona Lisa, using the same cosine-similarity idea.

In [ ]:
chunks = [
    "Bob is interested in the Mona Lisa, a famous painting on display at a museum in Paris.",
    "Alice works at Fenwick Restoration Studio as a Lead Conservator, restoring old paintings for various museums.",
    "Bob works at Meridian Analytics as a Senior Data Architect, founded in 2014.",
    "The Mona Lisa hangs in the Louvre museum, one of the most visited museums in the world.",
    "The painting is insured for $860 million, reflecting its priceless cultural significance.",
]

question = "What company does Bob work at?"

vectorizer = TfidfVectorizer().fit(chunks + [question])
chunk_vectors = vectorizer.transform(chunks)
question_vector = vectorizer.transform([question])
similarities = cosine_similarity(question_vector, chunk_vectors)[0]

ranked = pd.DataFrame({"chunk": chunks, "similarity": similarities}).sort_values("similarity", ascending=False)
display(ranked)

,chunk,similarity
2,Bob works at Meridian Analytics as a Senior Da...,0.121708
0,"Bob is interested in the Mona Lisa, a famous p...",0.111087
1,Alice works at Fenwick Restoration Studio as a...,0.044340
3,"The Mona Lisa hangs in the Louvre museum, one ...",0.000000
4,"The painting is insured for $860 million, refl...",0.000000


**Reading the output.** The most similar chunk is correctly the one about Bob's job — so for this question, plain text retrieval works fine. (We're using word-overlap similarity here, a simpler cousin of the embeddings you used in ChromaDB, so everything in this recap runs without downloading anything new.) Chunks that don't even mention Bob score near zero. This is traditional RAG working as intended: find the text that talks about the same things as the question, hand it to the model.

---
## Part 2 — The gap: "similar" isn't "correct"

Here's the problem. Try a *different* way of asking almost the same question — swap "insured for" with "insurance value" — and watch what happens.

In [ ]:
question2 = "What is the Mona Lisa's insurance value?"

question2_vector = vectorizer.transform([question2])
similarities2 = cosine_similarity(question2_vector, chunk_vectors)[0]

ranked2 = pd.DataFrame({"chunk": chunks, "similarity": similarities2}).sort_values("similarity", ascending=False)
display(ranked2)

,chunk,similarity
3,"The Mona Lisa hangs in the Louvre museum, one ...",0.397341
0,"Bob is interested in the Mona Lisa, a famous p...",0.392178
4,"The painting is insured for $860 million, refl...",0.190109
2,Bob works at Meridian Analytics as a Senior Da...,0.000000
1,Alice works at Fenwick Restoration Studio as a...,0.000000


**Reading the output.** The chunk that actually has the answer — "insured for $860 million" — drops to *third* place. The top-ranked chunk is just about the Louvre's location, which has nothing to do with insurance at all. Why? The insurance chunk says "insured for," and the question says "insurance value" — different words for the same idea. Text similarity only sees word overlap, not meaning, so a harmless rewording was enough to bury the one chunk that actually answers the question.

If a RAG system just grabs the top-ranked chunk and hands it to the model, this question gets answered from the *wrong* information — or the model has to guess anyway, right back to Part 0's problem.

> **Big idea.** Text retrieval finds chunks that *sound* like the question. It has no concept of a verified fact, a specific relationship, or which entity is actually being asked about. That's the gap a knowledge graph is built to close.

---
## Part 3 — Build a knowledge graph

Recall the Bob, Alice, and Mona Lisa triples from earlier. Let's extend that same small world with a handful of specific, made-up facts — a company, a job title, a dollar figure. These are facts **no model could possibly know from training**, which makes them perfect for testing whether grounding actually works: if the model gets these right, it's because it genuinely looked them up, not because it memorized them somewhere.

In [ ]:
triples = [
    ("Bob", "interestedIn", "Mona Lisa"),
    ("Alice", "interestedIn", "Mona Lisa"),
    ("Bob", "knows", "Alice"),
    ("Bob", "livesIn", "Paris"),
    ("Mona Lisa", "paintedBy", "Leonardo da Vinci"),
    ("Mona Lisa", "locatedIn", "Louvre"),
    ("Louvre", "locatedIn", "Paris"),
    ("Leonardo da Vinci", "bornIn", "Vinci"),
    # new, specific facts -- made up, and impossible for any model to know without retrieval
    ("Bob", "worksAt", "Meridian Analytics"),
    ("Bob", "hasRole", "Senior Data Architect"),
    ("Alice", "worksAt", "Fenwick Restoration Studio"),
    ("Alice", "hasRole", "Lead Conservator"),
    ("Bob", "metAliceAt", "the 2019 Louvre Digital Archives Conference"),
    ("Mona Lisa", "insuredFor", "$860 million"),
    ("Meridian Analytics", "foundedIn", "2014"),
]

G = nx.MultiDiGraph()
for s, p, o in triples:
    G.add_edge(s, o, key=p)

print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
for s, o, p in G.edges(keys=True):
    print(f"  {s} --{p}--> {o}")

14 nodes, 15 edges
  Bob --interestedIn--> Mona Lisa
  Bob --knows--> Alice
  Bob --livesIn--> Paris
  Bob --worksAt--> Meridian Analytics
  Bob --hasRole--> Senior Data Architect
  Bob --metAliceAt--> the 2019 Louvre Digital Archives Conference
  Mona Lisa --paintedBy--> Leonardo da Vinci
  Mona Lisa --locatedIn--> Louvre
  Mona Lisa --insuredFor--> $860 million
  Alice --interestedIn--> Mona Lisa
  Alice --worksAt--> Fenwick Restoration Studio
  Alice --hasRole--> Lead Conservator
  Leonardo da Vinci --bornIn--> Vinci
  Louvre --locatedIn--> Paris
  Meridian Analytics --foundedIn--> 2014


Same picture-style notation: `(Bob)-[:worksAt]->(Meridian Analytics)`. Fifteen facts, fourteen entities. This is deliberately tiny — the point is the *mechanism*, not the size of the graph. The same retrieval logic we're about to build would work on a graph of a million facts just as it does on fifteen.

---
## Part 4 — Retrieval: search the graph, not the text

Here's the two-step idea behind graph retrieval:

1. **Find which entities the question is actually about** — not by word-similarity, but by checking whether a real node's name appears in the question.
2. **Pull every fact directly connected to those entities.**

No embeddings, no "sounds similar" — just: which real things is this question asking about, and what do we actually know about them?

In [ ]:
def find_mentioned_entities(question, graph):
    question_lower = question.lower()
    return [node for node in graph.nodes if node.lower() in question_lower]

def retrieve_facts(question, graph, hops=1):
    entities = find_mentioned_entities(question, graph)
    facts = set()
    frontier = set(entities)
    for _ in range(hops):
        next_frontier = set()
        for e in frontier:
            for _, obj, pred in graph.out_edges(e, keys=True):
                facts.add((e, pred, obj)); next_frontier.add(obj)
            for subj, _, pred in graph.in_edges(e, keys=True):
                facts.add((subj, pred, e)); next_frontier.add(subj)
        frontier = next_frontier
    return entities, facts

question = "What company does Bob work at?"
entities, facts = retrieve_facts(question, G, hops=1)

print("entities found in the question:", entities)
print(f"\n{len(facts)} facts retrieved:")
for f in facts:
    print(" ", f)

entities found in the question: ['Bob']

6 facts retrieved:
  ('Bob', 'interestedIn', 'Mona Lisa')
  ('Bob', 'livesIn', 'Paris')
  ('Bob', 'worksAt', 'Meridian Analytics')
  ('Bob', 'knows', 'Alice')
  ('Bob', 'metAliceAt', 'the 2019 Louvre Digital Archives Conference')
  ('Bob', 'hasRole', 'Senior Data Architect')


**Reading the output.** The function found "Bob" mentioned in the question, then pulled every fact directly connected to Bob — his job, his company, his friendship, where he lives, what he's interested in, how he met Alice. All six facts are genuinely about Bob; nothing came from word-overlap guessing.

Compare this to Part 2's failure. There, a simple rewording ("insurance value" instead of "insured for") buried the right chunk. Here, rewording the question doesn't matter at all — "What company does Bob work at?" and "Where is Bob employed?" would both find the exact same entity, Bob, and pull the exact same facts.

> **Big idea.** Graph retrieval doesn't ask "which text sounds like this question?" It asks "which real entities is this question about, and what do we actually know about them?"

---
## Part 5 — Grounding: build a prompt that only uses retrieved facts

Retrieving the facts isn't enough on its own. We also have to **tell the model to use only what we gave it** — otherwise it might ignore our facts and fall back on guessing, right back to Part 0's problem. This is the grounding step: build a prompt that hands the model our retrieved facts and instructs it to answer only from them.

In [ ]:
def build_grounded_prompt(question, facts):
    facts_text = "\n".join(f"- {s} {p} {o}" for s, p, o in facts)
    system_message = (
        "You are a careful assistant. Answer the user's question using ONLY the facts "
        "listed below. If the facts do not contain the answer, say you don't have that "
        "information -- do not guess or use outside knowledge.\n\n"
        f"FACTS:\n{facts_text}"
    )
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question},
    ]

question = "What company does Bob work at?"
_, facts = retrieve_facts(question, G, hops=1)
messages = build_grounded_prompt(question, facts)

print("--- SYSTEM MESSAGE SENT TO THE MODEL ---")
print(messages[0]["content"])
print("\n--- USER QUESTION ---")
print(messages[1]["content"])

--- SYSTEM MESSAGE SENT TO THE MODEL ---
You are a careful assistant. Answer the user's question using ONLY the facts listed below. If the facts do not contain the answer, say you don't have that information -- do not guess or use outside knowledge.

FACTS:
- Bob interestedIn Mona Lisa
- Bob livesIn Paris
- Bob worksAt Meridian Analytics
- Bob knows Alice
- Bob metAliceAt the 2019 Louvre Digital Archives Conference
- Bob hasRole Senior Data Architect

--- USER QUESTION ---
What company does Bob work at?


**Reading the output.** This is exactly what gets sent to the model — nothing hidden. The facts are listed in plain text, and the instruction is explicit: use only these facts, and say so if the answer isn't here. The model never sees the whole graph, only the small slice Part 4's retrieval decided was relevant.

---
## Part 6 — Call the model, grounded

Now send that exact prompt to Groq's `openai/gpt-oss-120b` and see what comes back.

In [ ]:
grounded_answer = ask_model(messages)
print(grounded_answer)

Bob works at Meridian Analytics.


**Reading the output.** If your key is working, this should be a confident, correct answer — "Meridian Analytics" — because this time the model isn't guessing. It's reading the answer straight out of the facts we handed it. Compare this to Part 0's cold open, where the exact same question, with no facts attached, got either a refusal or an invented answer.

Same model. Same question. The only thing that changed is whether it had something real to look at.

---
## Part 7 — The payoff: side by side

Let's put Part 0's ungrounded attempt and Part 6's grounded answer right next to each other, on the same question, so the contrast is impossible to miss.

In [ ]:
question = "What company does Bob work at?"

ungrounded = ask_model([{"role": "user", "content": question}])
_, facts = retrieve_facts(question, G, hops=1)
grounded = ask_model(build_grounded_prompt(question, facts))

print("UNGROUNDED (no retrieval):")
print(" ", ungrounded)
print()
print("GROUNDED (facts retrieved from the graph first):")
print(" ", grounded)

UNGROUNDED (no retrieval):
  I’m sorry, but I can’t help with that.

GROUNDED (facts retrieved from the graph first):
  Bob works at Meridian Analytics.


Same question, same model, same point in time. The only difference is whether retrieval ran first. That's the entire mechanism of Graph RAG in one comparison: **retrieve, then ground, then answer** — instead of **just answer**.

---
## Part 8 — A quiet bug: retrieval that's too shallow

Here's a bug that's easy to miss, because it doesn't crash anything — it just quietly gives an incomplete answer. Ask a question that needs **two separate facts, one step apart from each other** in the graph.

In [ ]:
question = "Who painted the Mona Lisa, and where were they born?"

entities_1hop, facts_1hop = retrieve_facts(question, G, hops=1)
print(f"1 hop retrieves {len(facts_1hop)} facts:")
for f in facts_1hop: print(" ", f)
print("\ndoes this include where Leonardo was born?", any(p == "bornIn" for _, p, _ in facts_1hop))

1 hop retrieves 5 facts:
  ('Mona Lisa', 'insuredFor', '$860 million')
  ('Alice', 'interestedIn', 'Mona Lisa')
  ('Bob', 'interestedIn', 'Mona Lisa')
  ('Mona Lisa', 'locatedIn', 'Louvre')
  ('Mona Lisa', 'paintedBy', 'Leonardo da Vinci')

does this include where Leonardo was born? False


**Reading the output.** One hop out from "Mona Lisa" reaches "paintedBy Leonardo da Vinci" — but it stops there. Leonardo's own birthplace is a *second* hop away (Mona Lisa → Leonardo → Vinci), and 1-hop retrieval never reaches it. If we grounded the model on these facts alone, it could correctly answer "who painted it" but would have nothing to say about where they were born — not because the fact doesn't exist, but because retrieval never found it.

In [ ]:
entities_2hop, facts_2hop = retrieve_facts(question, G, hops=2)
print(f"2 hops retrieves {len(facts_2hop)} facts:")
for f in facts_2hop: print(" ", f)
print("\ndoes this include where Leonardo was born?", any(p == "bornIn" for _, p, _ in facts_2hop))

print()
answer_1hop = ask_model(build_grounded_prompt(question, facts_1hop))
answer_2hop = ask_model(build_grounded_prompt(question, facts_2hop))
print("Answer grounded on 1-hop facts:")
print(" ", answer_1hop)
print("\nAnswer grounded on 2-hop facts:")
print(" ", answer_2hop)

2 hops retrieves 14 facts:
  ('Bob', 'hasRole', 'Senior Data Architect')
  ('Alice', 'worksAt', 'Fenwick Restoration Studio')
  ('Mona Lisa', 'insuredFor', '$860 million')
  ('Alice', 'interestedIn', 'Mona Lisa')
  ('Bob', 'interestedIn', 'Mona Lisa')
  ('Bob', 'livesIn', 'Paris')
  ('Bob', 'knows', 'Alice')
  ('Leonardo da Vinci', 'bornIn', 'Vinci')
  ('Bob', 'worksAt', 'Meridian Analytics')
  ('Alice', 'hasRole', 'Lead Conservator')
  ('Louvre', 'locatedIn', 'Paris')
  ('Bob', 'metAliceAt', 'the 2019 Louvre Digital Archives Conference')
  ('Mona Lisa', 'locatedIn', 'Louvre')
  ('Mona Lisa', 'paintedBy', 'Leonardo da Vinci')

does this include where Leonardo was born? True

Answer grounded on 1-hop facts:
  The Mona Lisa was painted by Leonardo da Vinci.  
I don’t have any information about where Leonardo da Vinci was born.

Answer grounded on 2-hop facts:
  Leonardo da Vinci painted the Mona Lisa, and he was born in Vinci.


**Reading the output.** With 2 hops, Leonardo's birthplace (Vinci) is now in the retrieved facts, and the model can actually answer the whole question. With only 1 hop, the grounding prompt told the model to say it doesn't have the information — which is the *honest* response given what it was handed, but it's still an incomplete answer to what the user actually asked.

This is a genuinely easy bug to miss, because nothing errors out. The system runs, the prompt builds fine, the model follows instructions correctly — the retrieval step itself just didn't reach far enough. A structured prompt for this kind of task should include a line like:

```
CHECK FOR: does the question require chaining through more than one relationship?
      If so, confirm the retrieval hop count is large enough to reach every fact
      needed, not just the facts directly attached to the entities named in the
      question.
```

> **Big idea.** Grounding only works as well as retrieval does. A model that correctly refuses to guess is still giving an incomplete answer if retrieval stopped one hop too early.

---
## Part 9 — Where this leads

**Later this week:** the exact same system, with one piece swapped out. Right now, `retrieve_facts` searches a NetworkX graph sitting in this notebook's memory. Swap that one function for a Cypher query against a real Neo4j database, and everything downstream — the prompt construction, the grounding, the model call — stays identical. That's the payoff of building retrieval as its own clean step: the backend underneath it can change without touching anything else.

**Module 9 (Probabilistic Graphical Models).** A knowledge graph states facts with certainty. A PGM goes a step further, reasoning about *how confident* we should be in a conclusion when the available facts only partially support it.

---
## Key takeaways

1. **Traditional RAG retrieves text that sounds like the question.** It has no concept of a verified fact — just word overlap, which a simple rewording can fool.
2. **Graph retrieval searches for real entities and pulls their actual connections.** Rewording the question doesn't change which entities it's about.
3. **Grounding** means instructing the model to answer only from retrieved facts, and to say so honestly when the facts don't contain the answer.
4. **Same model, same question — the only variable that matters is whether retrieval ran first.** That's the entire mechanism of Graph RAG.
5. **Retrieval can quietly fall short without erroring.** A question that needs two connected facts requires retrieval to actually search two hops out, not one.
6. **Separating retrieval from everything else** is what makes swapping backends (NetworkX today, Neo4j later) possible without rebuilding the whole system.

---
## Activity and reflection

Add cells to complete these tasks and submit your notebook with your written answers.

**1. Add your own fact and question.** Add two or three new triples to the graph (anything you like — another person, another relationship). Write a question only your new facts can answer, and confirm the grounded model gets it right.

**2. Find a 2-hop question of your own.** Using the existing graph, write a question that needs at least two hops to fully answer (like Part 8's example), and show the 1-hop answer is incomplete while the 2-hop answer isn't.

**3. Audit for grounding leakage.** Ask the grounded model a question whose answer is *not* in your graph at all. Does it correctly say it doesn't know, or does it quietly fall back on outside knowledge despite the system prompt? Report what you found.

**4. Reflect (150-250 words).** Part 2 showed that a simple rewording can defeat text-based retrieval, while graph retrieval was unaffected by the same rewording. Give one example from your own field or interests where this distinction -- matching on *words* versus matching on *real entities and relationships* -- would matter in practice.

---
### References

- Lewis, P., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *NeurIPS.*
- Edge, D., et al. (2024). From local to global: A graph RAG approach to query-focused summarization. *Microsoft Research.*